# Causal Volatility Tokenizer: full experiment run (GPU)

Runs every experiment for the final report and resumes where it stopped if Colab disconnects.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Steps 1–6 run the six walk-forward folds, three seeds on Fold 1, the next-day horizon and the negative control (roughly 1–2 hours on a T4).
3. Look at the inflation table (step 6). The **2025 holdout** (step 7) is scored **once**, only after that review; the script refuses to score it a second time.
4. Step 8 zips `results/` for download. Send that zip back for the report.


### 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch the runtime to a GPU (Runtime > Change runtime type > T4 GPU)" 

### 2. Work folder on Google Drive (keeps progress if the session disconnects)

In [ ]:
USE_DRIVE = True   # set to False to work in /content only (lost on disconnect)
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/causal-vol-tokenizer'
else:
    WORK = '/content/causal-vol-tokenizer'
print(WORK)

### 3. Get the code
Until the PR is merged, the branch lives on the fork. After merging, switch to Devansh's repo and `main`.

In [ ]:
REPO_URL = 'https://github.com/diffused-orbital/causal-vol-tokenizer'   # after merge: https://github.com/devanshjangid2005/causal-vol-tokenizer
BRANCH   = 'fix/leak-audit-final'                                       # after merge: main
if os.path.exists(os.path.join(WORK, '.git')):
    !git -C "$WORK" fetch -q origin "$BRANCH" && git -C "$WORK" checkout -q "$BRANCH" && git -C "$WORK" pull -q --ff-only origin "$BRANCH"
else:
    !git clone -q --branch "$BRANCH" "$REPO_URL" "$WORK"
%cd $WORK
!git log --oneline -3

### 4. Install the two extra packages (PyTorch is already on Colab)

In [ ]:
!pip install -q arch pyarrow

### 5. Download Binance BTCUSDT 5-minute data, Jan 2021 – Dec 2025 (~1 minute; cached in `data/raw`)

In [ ]:
!python data/download_binance.py 2021 1 2025 12 | tail -2

### 6. Run all fold experiments
Re-run this cell after a disconnect: finished steps are skipped. Logs are in `results/logs/`.

In [ ]:
!python experiments/run_all.py

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/inflation_table_h12.md').read()))
for f in ['results/figures/inflation_all_folds_h12.png', 'results/figures/negative_control_oracle_floor.png']:
    if os.path.exists(f):
        display(Image(f))

### 7. Score the 2025 holdout (once)
Only run this after reviewing step 6. From here on the code is frozen: the commit below is recorded in `results/HOLDOUT_SCORED.json`.

In [ ]:
!git rev-parse HEAD
!python experiments/run_all.py --holdout
if os.path.exists('results/holdout_h12_leak_audit.json'):
    display(Image('results/figures/holdout_inflation_h12.png'))

### 8. Download the results

In [ ]:
!cd "$WORK" && rm -f /content/results.zip && zip -qr /content/results.zip results -x "results/logs/*"
from google.colab import files
files.download('/content/results.zip')